<a href="https://colab.research.google.com/github/AdrianCurellCruz/World_cup_prediction_using_ML_models/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import kagglehub
import os


# Download latest version using kaggle API
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
results_file_path = os.path.join(path, "results.csv")
df_all = pd.read_csv(results_file_path)


Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


Now we need to divide the dataset in two parts,one part is all the games from 1872 to 2026,just before the world cup

---



In [24]:
df_all['date']=pd.to_datetime(df_all['date'])
start_time='1872-11-30'
end_time='2026-06-10'
df = df_all[df_all['date'].between(start_time,end_time)]
#to check if we divided it correctly
print(df['date'].tail())

49400   2026-06-09
49401   2026-06-10
49402   2026-06-10
49403   2026-06-10
49404   2026-06-10
Name: date, dtype: datetime64[ns]


In [21]:
print(f'Dataset dimensions {df.shape}')
print(f'Number of instances: {df.shape[0]}')
print(f'Number of variables: {df.shape[1]}')
print(df.columns.tolist())

Dataset dimensions (49405, 9)
Number of instances: 49405
Number of variables: 9
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


We check if we have NAns in the data

In [25]:
null_counts = df.isnull().sum()
print("Null count per column:")
print(null_counts)

Null count per column:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64


We then extract all tournaments from the dataset and organize them into distinct hierarchical tiers. This process is essential for the subsequent development of a comprehensive Elo rating system.

In [28]:
tournaments = df['tournament'].unique().tolist()
print(tournaments)




['Friendly', 'British Home Championship', 'Évence Coppée Trophy', 'Muratti Vase', 'Copa Lipton', 'Copa Newton', 'Copa Premio Honor Argentino', 'Olympic Games', 'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games', 'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup', 'Open International Championship', 'Soccer Ashes', 'Copa Chevallier Boutell', 'Nordic Championship', 'Central European International Cup', 'Baltic Cup', 'Balkan Cup', 'Central American and Caribbean Games', 'FIFA World Cup', 'Copa Rio Branco', 'FIFA World Cup qualification', 'Bolivarian Games', 'CCCF Championship', 'NAFC Championship', 'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship', 'Copa del Pacífico', "Copa Bernardo O'Higgins", 'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup', 'African Cup of Nations', 'Copa Paz del Chaco', 'Merdeka Tournament', 'UEFA Euro qualification', 'Southeast Asian Peninsular Games', 'African Friendship Games', 'UEFA Euro', 'Windward Islands Tourn

Subsequently, we categorize the tournaments into distinct tiers based on their relative importance.

In [29]:
# Dictionary to store the weight for each tournament
tier_weights = {}

# Iterate through each tournament and assign the weight based on its category
for tournament in tournaments:
    # TIER 1: ONLY the World Cup final stage (Weight 1.0)
    if tournament == 'FIFA World Cup':
        tier_weights[tournament] = 1.0

    # TIER 2: Top continental competitions (Weight 0.9)
    # ONLY the finals of each confederation
    elif tournament in [
        'UEFA Euro',
        'Copa América',
        'African Cup of Nations',
        'AFC Asian Cup',
        'CONCACAF Gold Cup',
        'Oceania Nations Cup'
    ]:
        tier_weights[tournament] = 0.9

    # TIER 3: Intercontinental tournaments, Olympic Games, and elite historical cups (Weight 0.8)
    elif tournament in [
        'Olympic Games',
        'Confederations Cup',
        'CONMEBOL–UEFA Cup of Champions',
        'British Home Championship',
        'Central European International Cup',
        'Balkan Cup',
        'Nordic Championship'
    ]:
        tier_weights[tournament] = 0.8

    # TIER 4: Qualifiers and UEFA Nations League (Weight 0.6)
    elif 'qualification' in tournament.lower():
        tier_weights[tournament] = 0.6

    elif tournament in ['UEFA Nations League']:
        tier_weights[tournament] = 0.6

    # TIER 5: Sub-confederation regional tournaments (Weight 0.4)
    elif tournament in [
        'AFF Championship', 'EAFF Championship', 'SAFF Cup',
        'WAFF Championship', 'COSAFA Cup', 'CECAFA Cup',
        'UNCAF Cup', 'Gulf Cup', 'Arab Cup', 'CAFA Nations Cup',
        'ASEAN Championship', 'Gold Cup qualification',
        'CONCACAF Nations League'
    ]:
        tier_weights[tournament] = 0.4

    # Also detect by prefix (e.g., all tournaments starting with AFF, SAFF, etc.)
    elif tournament.startswith(('AFF', 'EAFF', 'SAFF', 'WAFF', 'COSAFA', 'CECAFA', 'UNCAF')):
        tier_weights[tournament] = 0.4

    # TIER 6: Friendlies, invitationals, minor cups, CONIFA, multi-sport games (Weight 0.2)
    else:
        tier_weights[tournament] = 0.2

# Display the results
print("Tournament weight assignment:")
for tourney, weight in sorted(tier_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {tourney}: {weight}")

# Check how many tournaments fall into each weight category
from collections import Counter
weight_counts = Counter(tier_weights.values())
print(f"\nWeight distribution: {dict(weight_counts)}")

Tournament weight assignment:
  FIFA World Cup: 1.0
  Copa América: 0.9
  AFC Asian Cup: 0.9
  African Cup of Nations: 0.9
  UEFA Euro: 0.9
  Oceania Nations Cup: 0.9
  British Home Championship: 0.8
  Olympic Games: 0.8
  Nordic Championship: 0.8
  Central European International Cup: 0.8
  Balkan Cup: 0.8
  CONMEBOL–UEFA Cup of Champions: 0.8
  Confederations Cup: 0.8
  FIFA World Cup qualification: 0.6
  AFC Asian Cup qualification: 0.6
  UEFA Euro qualification: 0.6
  African Cup of Nations qualification: 0.6
  CONCACAF Championship qualification: 0.6
  Copa América qualification: 0.6
  CFU Caribbean Cup qualification: 0.6
  Arab Cup qualification: 0.6
  Oceania Nations Cup qualification: 0.6
  COSAFA Cup qualification: 0.6
  Gold Cup qualification: 0.6
  AFF Championship qualification: 0.6
  AFC Challenge Cup qualification: 0.6
  UEFA Nations League: 0.6
  CONCACAF Nations League qualification: 0.6
  CONIFA World Cup qualification: 0.6
  CONIFA World Football Cup qualification: 0.6